# DotMatch Quickstart

Use DotMatch when you already know the short DNA sequences expected in a read window. This quickstart shows the Python API and the CLI on a tiny CRISPR guide-counting fixture.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile

import dotmatch
targets = ["ACGT", "TTTT", "GGGG"]
reads = ["ACGT", "ACGA", "CCCC"]
results = dotmatch.assign(reads, targets, k=1)
[(read, result.target_index, result.best_distance, result.status) for read, result in zip(reads, results)]

Status values are exported constants: `MATCH_UNIQUE`, `MATCH_AMBIGUOUS`, `MATCH_NONE`, and `MATCH_INVALID`.

In [ ]:
status_names = {
    dotmatch.MATCH_UNIQUE: "unique",
    dotmatch.MATCH_AMBIGUOUS: "ambiguous",
    dotmatch.MATCH_NONE: "none",
    dotmatch.MATCH_INVALID: "invalid",
}
[
    {
        "read": read,
        "target_index": result.target_index,
        "best_distance": result.best_distance,
        "status": status_names[result.status],
    }
    for read, result in zip(reads, results)
]

The CLI streams FASTQ/FASTQ.gz and writes count, assignment, and summary files.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "demo-data" / "crispr_guides.tsv").exists():
            return path
    raise FileNotFoundError("could not find demo-data/crispr_guides.tsv")

root = find_repo_root(Path.cwd())
target_file = root / "demo-data" / "crispr_guides.tsv"
read_file = root / "demo-data" / "reads.fastq"

with tempfile.TemporaryDirectory(prefix="dotmatch-quickstart-") as tmp:
    out_dir = Path(tmp)
    counts = out_dir / "counts.tsv"
    summary = out_dir / "summary.json"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "dotmatch.cli",
            "count",
            "--targets",
            str(target_file),
            "--reads",
            str(read_file),
            "--target-start",
            "0",
            "--target-length",
            "4",
            "--k",
            "1",
            "--out",
            str(counts),
            "--summary",
            str(summary),
        ],
        check=True,
        env={**os.environ, "DOTMATCH_PYTHON_NO_DELEGATE": "1"},
    )
    with counts.open(encoding="utf-8") as fh:
        count_table = list(csv.DictReader(fh, delimiter="\t"))
    run_summary = json.loads(summary.read_text(encoding="utf-8"))

count_table

In [ ]:
assert run_summary["total_reads"] == 3
assert run_summary["assigned_unique"] == 2
assert run_summary["unmatched"] == 1
run_summary